# Publication Panels

This notebook develops the publication-style cross-species figures using outputs in `cross_species_consensus_v3`.

Current focus:
- Panel 1: differential accessibility volcano panels
- Shared provenance manifest for downstream panels

## Contrast Design: Primary vs. Secondary

**Primary Contrasts** (`Pair_Human_vs_Chimp`, `Pair_Human_vs_Gorilla`):
- Direct pairwise comparisons between humans and our two closest living relatives
- These contrasts are most reliable for identifying species-specific regulatory divergence
- Changes reflect evolutionary shifts since speciation from our common ancestors

**Secondary Contrasts** (`Pair_Human_vs_Bonobo`, `Div_Human_vs_Apes`):
- `Pair_Human_vs_Bonobo`: More distant pairwise comparison (included for completeness)
- `Div_Human_vs_Apes`: Divergence signal comparing humans as one group vs. all apes combined
- These are secondary checks and may show broader evolutionary patterns but with less specificity

**Why split them?**
Primary contrasts provide the strongest statistical power and clearest biological signal for understanding what changed in the human lineage specifically. Secondary contrasts help validate findings and detect patterns that may be shared across multiple species divergences.

In [2]:
from __future__ import annotations

import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

PROJECT_ROOT = Path('/home/jjanssens/jjans/analysis/adult_intestine/peaks/peak_calling/atac_pipeline')
OUTPUT_ROOT = Path('/home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3')
BRANCH_RESULTS = OUTPUT_ROOT / '13_deseq2_R_pseudobulk' / 'differential_results' / 'evolutionary_branches'
ANNOTATION_FILE = OUTPUT_ROOT / '07_master_annotation' / 'master_annotation_final.tsv'
PANEL1_OUTDIR = OUTPUT_ROOT / '21_publication_panels' / 'panel1_da_volcanoes'
PANEL1_OUTDIR.mkdir(parents=True, exist_ok=True)

FOREGROUND_CELL_TYPES = ['Macrophages', 'T cells', 'Enterocytes']
PRIMARY_CONTRASTS = ['Pair_Human_vs_Chimp', 'Pair_Human_vs_Gorilla']
SECONDARY_CONTRASTS = ['Pair_Human_vs_Bonobo', 'Div_Human_vs_Apes']
ALL_CONTRASTS = PRIMARY_CONTRASTS + SECONDARY_CONTRASTS
PADJ_THRESH = 0.05
LFC_THRESH = 1.0
LABELS_PER_DIRECTION = 8

VOLCANO_COLORS = {
    'positive': '#c23b22',
    'negative': '#2563eb',
    'neutral': '#c7c7c7',
}

In [3]:
def celltype_to_dirname(cell_type: str) -> str:
    return re.sub(r'[^A-Za-z0-9]', '.', cell_type)


def sanitize_name(value: str) -> str:
    value = str(value).strip()
    value = re.sub(r'\s+', '_', value)
    return re.sub(r'[^A-Za-z0-9_.+-]', '_', value)


def parse_contrast_name(contrast: str) -> dict[str, str]:
    base = contrast
    for prefix in ('Pair_', 'Div_', 'ILS_'):
        if base.startswith(prefix):
            base = base[len(prefix):]
            break
    if '_vs_' in base:
        lhs, rhs = base.split('_vs_', 1)
    else:
        lhs, rhs = base, 'Others'
    lhs_display = lhs.replace('_', ' ')
    rhs_display = rhs.replace('_', ' ')
    return {
        'contrast': contrast,
        'display': f'{lhs_display} vs {rhs_display}',
        'positive_label': f'{lhs_display}-high',
        'negative_label': f'{rhs_display}-high',
    }


def load_annotation(annotation_file: Path) -> pd.DataFrame:
    annotation_df = pd.read_csv(annotation_file, sep='\t', low_memory=False)
    keep_columns = ['peak_id']
    if 'Human_gene' in annotation_df.columns:
        keep_columns.append('Human_gene')
    if 'Human_gene_dist' in annotation_df.columns:
        keep_columns.append('Human_gene_dist')
    annotation_df = annotation_df[keep_columns].copy()
    annotation_df = annotation_df.rename(columns={'Human_gene': 'nearest_gene', 'Human_gene_dist': 'gene_distance'})
    if 'nearest_gene' not in annotation_df.columns:
        annotation_df['nearest_gene'] = pd.NA
    if 'gene_distance' not in annotation_df.columns:
        annotation_df['gene_distance'] = pd.NA
    annotation_df['nearest_gene'] = annotation_df['nearest_gene'].replace({'.': pd.NA, '': pd.NA}).astype('string')
    return annotation_df.drop_duplicates(subset=['peak_id'])


def load_deseq_result(results_root: Path, cell_type: str, contrast: str) -> pd.DataFrame:
    result_path = results_root / celltype_to_dirname(cell_type) / f'{contrast}.csv'
    df = pd.read_csv(result_path)
    id_col = 'peak_id' if 'peak_id' in df.columns else df.columns[0]
    return df.rename(columns={id_col: 'peak_id'})


def prepare_volcano_df(result_df: pd.DataFrame, annotation_df: pd.DataFrame, contrast_info: dict[str, str]) -> pd.DataFrame:
    plot_df = result_df.merge(annotation_df, on='peak_id', how='left')
    plot_df = plot_df.dropna(subset=['padj', 'log2FoldChange']).copy()
    plot_df['padj_safe'] = plot_df['padj'].clip(lower=1e-300)
    plot_df['neg_log10_padj'] = -np.log10(plot_df['padj_safe'])
    plot_df['rank_score'] = plot_df['log2FoldChange'].abs() * plot_df['neg_log10_padj']
    plot_df['direction'] = 'Not significant'
    positive_mask = (plot_df['padj'] < PADJ_THRESH) & (plot_df['log2FoldChange'] >= LFC_THRESH)
    negative_mask = (plot_df['padj'] < PADJ_THRESH) & (plot_df['log2FoldChange'] <= -LFC_THRESH)
    plot_df.loc[positive_mask, 'direction'] = contrast_info['positive_label']
    plot_df.loc[negative_mask, 'direction'] = contrast_info['negative_label']
    plot_df['label_gene'] = plot_df['nearest_gene'].fillna(plot_df['peak_id']).astype(str)
    plot_df.loc[plot_df['label_gene'].isin(['<NA>', 'nan', '.']), 'label_gene'] = plot_df['peak_id']
    return plot_df


def select_top_labels(plot_df: pd.DataFrame, contrast_info: dict[str, str], n_per_direction: int = LABELS_PER_DIRECTION) -> pd.DataFrame:
    label_frames = []
    for direction in (contrast_info['positive_label'], contrast_info['negative_label']):
        direction_df = plot_df[plot_df['direction'] == direction].copy()
        if direction_df.empty:
            continue
        direction_df = direction_df.sort_values(['rank_score', 'baseMean'], ascending=[False, False])
        direction_df = direction_df.drop_duplicates(subset=['label_gene'])
        label_frames.append(direction_df.head(n_per_direction))
    if not label_frames:
        return plot_df.iloc[0:0].copy()
    return pd.concat(label_frames, ignore_index=False)


def build_manifest(results_root: Path, annotation_df: pd.DataFrame, cell_types: list[str], contrasts: list[str]) -> pd.DataFrame:
    records = []
    for cell_type in cell_types:
        for contrast in contrasts:
            result_path = results_root / celltype_to_dirname(cell_type) / f'{contrast}.csv'
            contrast_info = parse_contrast_name(contrast)
            record = {
                'panel': 'differential_accessibility_volcano',
                'cell_type': cell_type,
                'contrast': contrast,
                'contrast_display': contrast_info['display'],
                'result_path': str(result_path),
                'exists': result_path.exists(),
            }
            if result_path.exists():
                result_df = load_deseq_result(results_root, cell_type, contrast)
                plot_df = prepare_volcano_df(result_df, annotation_df, contrast_info)
                record['n_rows'] = len(plot_df)
                record['n_positive'] = int((plot_df['direction'] == contrast_info['positive_label']).sum())
                record['n_negative'] = int((plot_df['direction'] == contrast_info['negative_label']).sum())
                record['min_padj'] = float(plot_df['padj'].min())
                record['max_abs_log2fc'] = float(plot_df['log2FoldChange'].abs().max())
            records.append(record)
    return pd.DataFrame(records)


def infer_axis_limits(plot_frames: list[pd.DataFrame]) -> tuple[float, float]:
    merged = pd.concat(plot_frames, ignore_index=True)
    x_max = float(np.nanquantile(np.abs(merged['log2FoldChange']), 0.995))
    y_max = float(np.nanquantile(merged['neg_log10_padj'], 0.995))
    x_max = max(x_max, LFC_THRESH * 1.5, 2.0) * 1.05
    y_max = max(y_max, -math.log10(PADJ_THRESH) * 1.5, 5.0) * 1.05
    return x_max, y_max


def annotate_points(ax, label_df: pd.DataFrame) -> None:
    for idx, (_, row) in enumerate(label_df.iterrows()):
        x_offset = 10 if row['log2FoldChange'] >= 0 else -10
        y_offset = 6 + (idx % 5) * 5
        ha = 'left' if row['log2FoldChange'] >= 0 else 'right'
        ax.annotate(
            row['label_gene'],
            xy=(row['log2FoldChange'], row['neg_log10_padj']),
            xytext=(x_offset, y_offset),
            textcoords='offset points',
            fontsize=7,
            fontstyle='italic',
            ha=ha,
            va='bottom',
            arrowprops={'arrowstyle': '-', 'lw': 0.6, 'color': '#666666', 'alpha': 0.7},
        )


def plot_volcano_panel(results_root: Path, annotation_df: pd.DataFrame, contrast: str, cell_types: list[str], outdir: Path) -> dict[str, Path]:
    contrast_info = parse_contrast_name(contrast)
    panel_entries = []
    plot_frames = []
    label_tables = []

    for cell_type in cell_types:
        result_df = load_deseq_result(results_root, cell_type, contrast)
        plot_df = prepare_volcano_df(result_df, annotation_df, contrast_info)
        label_df = select_top_labels(plot_df, contrast_info).assign(cell_type=cell_type, contrast=contrast)
        panel_entries.append((cell_type, plot_df, label_df))
        plot_frames.append(plot_df)
        label_tables.append(label_df)

    x_lim, y_lim = infer_axis_limits(plot_frames)
    fig, axes = plt.subplots(1, len(cell_types), figsize=(5.1 * len(cell_types), 5.4), sharex=True, sharey=True)
    if len(cell_types) == 1:
        axes = [axes]

    for ax, (cell_type, plot_df, label_df) in zip(axes, panel_entries):
        neutral_df = plot_df[plot_df['direction'] == 'Not significant']
        positive_df = plot_df[plot_df['direction'] == contrast_info['positive_label']]
        negative_df = plot_df[plot_df['direction'] == contrast_info['negative_label']]

        ax.scatter(neutral_df['log2FoldChange'], neutral_df['neg_log10_padj'], s=8, c=VOLCANO_COLORS['neutral'], alpha=0.55, linewidths=0)
        ax.scatter(positive_df['log2FoldChange'], positive_df['neg_log10_padj'], s=12, c=VOLCANO_COLORS['positive'], alpha=0.8, linewidths=0)
        ax.scatter(negative_df['log2FoldChange'], negative_df['neg_log10_padj'], s=12, c=VOLCANO_COLORS['negative'], alpha=0.8, linewidths=0)

        annotate_points(ax, label_df)
        ax.axvline(LFC_THRESH, linestyle='--', color='#555555', linewidth=0.9)
        ax.axvline(-LFC_THRESH, linestyle='--', color='#555555', linewidth=0.9)
        ax.axhline(-math.log10(PADJ_THRESH), linestyle='--', color='#555555', linewidth=0.9)
        ax.set_xlim(-x_lim, x_lim)
        ax.set_ylim(0, y_lim)
        ax.set_title(f"{cell_type}\n{len(positive_df)} {contrast_info['positive_label']} | {len(negative_df)} {contrast_info['negative_label']}", fontsize=11, fontweight='bold')
        ax.set_xlabel('log2 fold change')

    axes[0].set_ylabel('-log10 adjusted p-value')
    fig.suptitle(f"Differential accessibility: {contrast_info['display']}", fontsize=14, fontweight='bold', y=1.02)
    fig.tight_layout()

    stem = sanitize_name(contrast)
    pdf_path = outdir / f'{stem}_volcano_panel.pdf'
    png_path = outdir / f'{stem}_volcano_panel.png'
    labels_path = outdir / f'{stem}_volcano_labels.tsv'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    labels_df = pd.concat(label_tables, ignore_index=True)
    labels_df.to_csv(labels_path, sep='\t', index=False)
    return {'pdf': pdf_path, 'png': png_path, 'labels': labels_path}

In [19]:
def y_transform(raw_values: pd.Series | np.ndarray) -> np.ndarray:
    """Smoothly compress -log10(padj) values without hard clipping plateaus."""
    arr = np.asarray(raw_values, dtype=float)
    return np.log10(1.0 + arr)


def is_nice_gene_name(gene_name: str) -> bool:
    if gene_name is None:
        return False
    s = str(gene_name).strip()
    if s in {'', '.', 'nan', '<NA>'}:
        return False
    if re.match(r'^ENS[A-Z0-9]*', s, flags=re.IGNORECASE):
        return False
    if re.match(r'^[0-9._-]+$', s):
        return False
    if re.match(r'^[A-Za-z][A-Za-z0-9._-]*$', s) is None:
        return False
    return True


def prepare_volcano_df(result_df: pd.DataFrame, annotation_df: pd.DataFrame, contrast_info: dict[str, str]) -> pd.DataFrame:
    """Prepare volcano data with human-positive orientation and data-driven p-value flooring."""
    plot_df = result_df.merge(annotation_df, on='peak_id', how='left')
    plot_df = plot_df.dropna(subset=['padj', 'log2FoldChange']).copy()

    # Enforce talk convention: Human-up on positive x-axis for Human-vs-* contrasts.
    if contrast_info['positive_label'].startswith('Human-'):
        plot_df['log2FoldChange'] = -plot_df['log2FoldChange']

    # Replace exact zeros using a data-driven floor to avoid artificial jumps at 300.
    non_zero = plot_df.loc[plot_df['padj'] > 0, 'padj']
    if len(non_zero) > 0:
        data_floor = float(non_zero.min()) * 0.5
    else:
        data_floor = 1e-50
    data_floor = max(data_floor, 1e-120)

    plot_df['padj_safe'] = plot_df['padj'].where(plot_df['padj'] > 0, data_floor)
    plot_df['neg_log10_padj'] = -np.log10(plot_df['padj_safe'])
    plot_df['neg_log10_padj_plot'] = y_transform(plot_df['neg_log10_padj'])
    plot_df['rank_score'] = plot_df['log2FoldChange'].abs() * plot_df['neg_log10_padj']

    plot_df['direction'] = 'Not significant'
    positive_mask = (plot_df['padj'] < PADJ_THRESH) & (plot_df['log2FoldChange'] > 0)
    negative_mask = (plot_df['padj'] < PADJ_THRESH) & (plot_df['log2FoldChange'] < 0)
    plot_df.loc[positive_mask, 'direction'] = contrast_info['positive_label']
    plot_df.loc[negative_mask, 'direction'] = contrast_info['negative_label']

    plot_df['label_gene'] = plot_df['nearest_gene'].fillna(plot_df['peak_id']).astype(str)
    plot_df.loc[plot_df['label_gene'].isin(['<NA>', 'nan', '.']), 'label_gene'] = plot_df['peak_id']
    return plot_df


def select_top_labels(plot_df: pd.DataFrame, contrast_info: dict[str, str], n_per_direction: int = LABELS_PER_DIRECTION) -> pd.DataFrame:
    label_frames = []
    # Avoid top-edge label crowding by excluding the extreme 0.2% of y-values.
    y_label_cap = float(np.nanquantile(plot_df['neg_log10_padj'], 0.998))

    for direction in (contrast_info['positive_label'], contrast_info['negative_label']):
        direction_df = plot_df[plot_df['direction'] == direction].copy()
        if direction_df.empty:
            continue
        direction_df = direction_df[direction_df['label_gene'].map(is_nice_gene_name)]
        direction_df = direction_df[direction_df['neg_log10_padj'] <= y_label_cap]
        if direction_df.empty:
            continue
        direction_df = direction_df.sort_values(['rank_score', 'baseMean'], ascending=[False, False])
        direction_df = direction_df.drop_duplicates(subset=['label_gene'])
        label_frames.append(direction_df.head(n_per_direction))
    if not label_frames:
        return plot_df.iloc[0:0].copy()
    return pd.concat(label_frames, ignore_index=False)


def annotate_points(ax, label_df: pd.DataFrame) -> None:
    """Anchor labels exactly at plotted coordinates."""
    for idx, (_, row) in enumerate(label_df.iterrows()):
        x_offset = 9 if row['log2FoldChange'] >= 0 else -9
        y_offset = 4 + (idx % 4) * 4
        ha = 'left' if row['log2FoldChange'] >= 0 else 'right'
        ax.annotate(
            row['label_gene'],
            xy=(row['log2FoldChange'], row['neg_log10_padj_plot']),
            xytext=(x_offset, y_offset),
            textcoords='offset points',
            fontsize=7,
            fontstyle='italic',
            ha=ha,
            va='bottom',
            arrowprops={'arrowstyle': '-', 'lw': 0.6, 'color': '#666666', 'alpha': 0.75},
        )


def infer_axis_limits(plot_frames: list[pd.DataFrame]) -> tuple[float, float]:
    merged = pd.concat(plot_frames, ignore_index=True)
    x_max = float(np.nanquantile(np.abs(merged['log2FoldChange']), 0.999))
    y_max = float(np.nanquantile(merged['neg_log10_padj_plot'], 0.999))
    x_max = max(x_max, 4.0) * 1.28
    y_max = max(y_max, y_transform(np.array([10.0]))[0]) * 1.15
    return x_max, y_max


def set_y_ticks(ax, y_lim: float) -> None:
    """Show interpretable tick labels in raw -log10(padj) units on transformed axis."""
    raw_ticks = np.array([1, 2, 3, 5, 8, 12, 20, 30, 50, 80, 120], dtype=float)
    transformed_ticks = y_transform(raw_ticks)
    keep = transformed_ticks <= y_lim
    ax.set_yticks(transformed_ticks[keep])
    ax.set_yticklabels([str(int(v)) for v in raw_ticks[keep]])


def plot_volcano_panel(results_root: Path, annotation_df: pd.DataFrame, contrast: str, cell_types: list[str], outdir: Path, write_labels: bool = True) -> dict[str, Path]:
    """Plot volcano panel with optional gene label annotations.
    
    Args:
        write_labels: If False, omit gene name annotations for cleaner appearance.
    """
    contrast_info = parse_contrast_name(contrast)
    panel_entries = []
    plot_frames = []
    label_tables = []

    for cell_type in cell_types:
        result_df = load_deseq_result(results_root, cell_type, contrast)
        plot_df = prepare_volcano_df(result_df, annotation_df, contrast_info)
        label_df = select_top_labels(plot_df, contrast_info).assign(cell_type=cell_type, contrast=contrast)
        panel_entries.append((cell_type, plot_df, label_df))
        plot_frames.append(plot_df)
        label_tables.append(label_df)

    x_lim, y_lim = infer_axis_limits(plot_frames)
    fig, axes = plt.subplots(1, len(cell_types), figsize=(5.4 * len(cell_types), 5.6), sharex=True, sharey=True)
    if len(cell_types) == 1:
        axes = [axes]

    colors = {
        'neutral': '#e8e8e8',
        'positive': '#d1495b',
        'negative': '#2b59c3',
    }

    for ax, (cell_type, plot_df, label_df) in zip(axes, panel_entries):
        neutral_df = plot_df[plot_df['direction'] == 'Not significant']
        positive_df = plot_df[plot_df['direction'] == contrast_info['positive_label']]
        negative_df = plot_df[plot_df['direction'] == contrast_info['negative_label']]

        ax.scatter(
            neutral_df['log2FoldChange'], neutral_df['neg_log10_padj_plot'],
            s=6, c=colors['neutral'], alpha=0.55, linewidths=0, rasterized=True
        )
        ax.scatter(
            negative_df['log2FoldChange'], negative_df['neg_log10_padj_plot'],
            s=11, c=colors['negative'], alpha=0.82, linewidths=0, rasterized=True
        )
        ax.scatter(
            positive_df['log2FoldChange'], positive_df['neg_log10_padj_plot'],
            s=11, c=colors['positive'], alpha=0.82, linewidths=0, rasterized=True
        )

        if write_labels:
            annotate_points(ax, label_df)
        ax.axhline(y_transform(np.array([-math.log10(PADJ_THRESH)]))[0], linestyle='--', color='#666666', linewidth=0.8)
        ax.axvline(0, linestyle='-', color='#444444', linewidth=0.9, alpha=0.85)
        ax.axvline(LFC_THRESH, linestyle='--', color='#999999', linewidth=0.7, alpha=0.8)
        ax.axvline(-LFC_THRESH, linestyle='--', color='#999999', linewidth=0.7, alpha=0.8)

        ax.set_xlim(-x_lim, x_lim)
        ax.set_ylim(0, y_lim)
        set_y_ticks(ax, y_lim)
        ax.set_title(
            f"{cell_type}\n{len(positive_df)} {contrast_info['positive_label']} | {len(negative_df)} {contrast_info['negative_label']}",
            fontsize=10.5,
            fontweight='bold'
        )
        ax.set_xlabel('log2(fold change)')
        ax.grid(alpha=0.12, linewidth=0.4)

    axes[0].set_ylabel('-log10(adj. p-value)')
    fig.suptitle(
        f"Differential accessibility: {contrast_info['display']}\nColored dots: padj < 0.05 (blue = {contrast_info['negative_label']}, red = {contrast_info['positive_label']})",
        fontsize=13,
        fontweight='bold',
        y=1.03,
    )
    fig.tight_layout()

    stem = sanitize_name(contrast)
    if write_labels:
        pdf_path = outdir / f'{stem}_volcano_panel.pdf'
        png_path = outdir / f'{stem}_volcano_panel.png'
    else:
        pdf_path = outdir / f'{stem}_volcano_panel_no_labels.pdf'
        png_path = outdir / f'{stem}_volcano_panel_no_labels.png'
    labels_path = outdir / f'{stem}_volcano_labels.tsv'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    labels_df = pd.concat(label_tables, ignore_index=True)
    labels_df.to_csv(labels_path, sep='\t', index=False)
    return {'pdf': pdf_path, 'png': png_path, 'labels': labels_path}

In [20]:
annotation_df = load_annotation(ANNOTATION_FILE)
manifest_df = build_manifest(BRANCH_RESULTS, annotation_df, FOREGROUND_CELL_TYPES, ALL_CONTRASTS)
manifest_path = PANEL1_OUTDIR / 'da_volcano_manifest.tsv'
manifest_df.to_csv(manifest_path, sep='\t', index=False)
manifest_df

,panel,cell_type,contrast,contrast_display,result_path,exists,n_rows,n_positive,n_negative,min_padj,max_abs_log2fc
0,differential_accessibility_volcano,Macrophages,Pair_Human_vs_Chimp,Human vs Chimp,/home/jjanssens/jjans/analysis/adult_intestine...,True,132015,9429,13333,0.0,11.101424
1,differential_accessibility_volcano,Macrophages,Pair_Human_vs_Gorilla,Human vs Gorilla,/home/jjanssens/jjans/analysis/adult_intestine...,True,126418,12148,14333,0.0,13.368342
2,differential_accessibility_volcano,Macrophages,Pair_Human_vs_Bonobo,Human vs Bonobo,/home/jjanssens/jjans/analysis/adult_intestine...,True,85309,1149,1044,0.0,12.152746
3,differential_accessibility_volcano,Macrophages,Div_Human_vs_Apes,Human vs Apes,/home/jjanssens/jjans/analysis/adult_intestine...,True,129726,6829,17328,0.0,9.626305
4,differential_accessibility_volcano,T cells,Pair_Human_vs_Chimp,Human vs Chimp,/home/jjanssens/jjans/analysis/adult_intestine...,True,241150,10802,16031,0.0,11.616084
5,differential_accessibility_volcano,T cells,Pair_Human_vs_Gorilla,Human vs Gorilla,/home/jjanssens/jjans/analysis/adult_intestine...,True,138810,19023,18376,0.0,13.599262
6,differential_accessibility_volcano,T cells,Pair_Human_vs_Bonobo,Human vs Bonobo,/home/jjanssens/jjans/analysis/adult_intestine...,True,150186,6065,6547,0.0,12.084355
7,differential_accessibility_volcano,T cells,Div_Human_vs_Apes,Human vs Apes,/home/jjanssens/jjans/analysis/adult_intestine...,True,255242,8286,23326,0.0,10.193294
8,differential_accessibility_volcano,Enterocytes,Pair_Human_vs_Chimp,Human vs Chimp,/home/jjanssens/jjans/analysis/adult_intestine...,True,140191,441,11023,0.0,12.829304
9,differential_accessibility_volcano,Enterocytes,Pair_Human_vs_Gorilla,Human vs Gorilla,/home/jjanssens/jjans/analysis/adult_intestine...,True,54894,2700,3637,0.0,10.296525


In [21]:
panel_outputs = []
for contrast in ALL_CONTRASTS:
    # Generate labeled version
    panel_outputs.append({
        'contrast': contrast, 
        'version': 'with_labels',
        **plot_volcano_panel(BRANCH_RESULTS, annotation_df, contrast, FOREGROUND_CELL_TYPES, PANEL1_OUTDIR, write_labels=True)
    })
    # Generate label-free version
    panel_outputs.append({
        'contrast': contrast,
        'version': 'no_labels', 
        **plot_volcano_panel(BRANCH_RESULTS, annotation_df, contrast, FOREGROUND_CELL_TYPES, PANEL1_OUTDIR, write_labels=False)
    })

panel_outputs_df = pd.DataFrame(panel_outputs)
panel_outputs_df

,contrast,version,pdf,png,labels
0,Pair_Human_vs_Chimp,with_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
1,Pair_Human_vs_Chimp,no_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
2,Pair_Human_vs_Gorilla,with_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
3,Pair_Human_vs_Gorilla,no_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
4,Pair_Human_vs_Bonobo,with_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
5,Pair_Human_vs_Bonobo,no_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
6,Div_Human_vs_Apes,with_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...
7,Div_Human_vs_Apes,no_labels,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...,/home/jjanssens/jjans/analysis/adult_intestine...


## Next panels

Planned additions in this notebook:
- GREAT term-by-cell-type heatmap
- Distance-to-human overview
- IBD locus tracks
- Macrophage motif spotlight

# Panel 2: GO Enrichment Analysis

Gene Ontology (GO) enrichment identifies biological processes and functional categories associated with human-divergent peaks. We'll visualize the top enriched biological process terms across cell types.

**Data availability note:** Pre-computed enrichment results are available for Enterocytes. For Macrophages and T cells, enrichment would require access to web-based tools (GREAT API) or additional computational resources. The current panel focuses on Enterocytes as a representative example; additional cell types can be added by running enrichment analysis independently.

In [22]:
# Panel 2 Configuration
PANEL2_OUTDIR = OUTPUT_ROOT / '21_publication_panels' / 'panel2_great_enrichment'
PANEL2_OUTDIR.mkdir(parents=True, exist_ok=True)

# GREAT analysis parameters
GREAT_FOCUS_CONTRAST = 'Div_Human_vs_Apes'  # Human divergence peaks
GREAT_BG_CONTRASTS = ['Pair_Human_vs_Chimp', 'Pair_Human_vs_Gorilla', 'Pair_Human_vs_Bonobo']  # For comparison

# Peak size filtering for GREAT (sometimes better with size cutoff)
MIN_PEAK_SIZE = 200  # bp
MAX_PEAK_SIZE = None  # No upper limit

In [23]:
def extract_significant_peaks(result_df: pd.DataFrame, annotation_df: pd.DataFrame, direction: str = 'positive') -> pd.DataFrame:
    """Extract significant peaks with genomic coordinates for enrichment analysis.
    
    Args:
        direction: 'positive' or 'negative' for up/down peaks, 'all' for all significant
    """
    sig_df = result_df[result_df['padj'] < PADJ_THRESH].copy()
    
    if direction == 'positive':
        sig_df = sig_df[sig_df['log2FoldChange'] > 0]
    elif direction == 'negative':
        sig_df = sig_df[sig_df['log2FoldChange'] < 0]
    
    # Merge with annotation to get peak coordinates (if available)
    sig_df = sig_df.merge(annotation_df[['peak_id']], on='peak_id', how='inner')
    return sig_df


def load_peak_coordinates(peak_ids: list[str], ref_genome: str = 'hg38') -> pd.DataFrame:
    """Load genomic coordinates for peaks from master annotation or feature file."""
    # For now, create a minimal coordinate dataframe
    # In production, this would read from a peak file or GTF
    coords = []
    for peak_id in peak_ids:
        # Example: peak_id format like "chr1:1000-2000"
        if ':' in peak_id and '-' in peak_id:
            try:
                chrom, region = peak_id.split(':')
                start, end = region.split('-')
                coords.append({
                    'peak_id': peak_id,
                    'chrom': chrom,
                    'start': int(start),
                    'end': int(end),
                    'size': int(end) - int(start)
                })
            except:
                pass
    return pd.DataFrame(coords)


def query_great_api(peak_coords: pd.DataFrame, species: str = 'hg38', request_id: str = None) -> dict:
    """Query GREAT API using uploaded BED file.
    
    Returns dict with enrichment results or None if unable to query.
    """
    try:
        import requests
        from io import StringIO
        
        # Prepare BED format
        bed_content = StringIO()
        for _, row in peak_coords.iterrows():
            bed_content.write(f"{row['chrom']}\t{row['start']}\t{row['end']}\n")
        
        # GREAT API endpoint
        url = "http://great.stanford.edu/public-api/v3/analyze/submit"
        
        files = {'bed': ('peaks.bed', bed_content.getvalue())}
        data = {
            'species': species,
            'requestName': request_id or 'panel2_analysis'
        }
        
        resp = requests.post(url, files=files, data=data, timeout=30)
        if resp.status_code == 200:
            result = resp.json()
            return result
        else:
            print(f"GREAT API error: {resp.status_code}")
            return None
    except Exception as e:
        print(f"Cannot query GREAT: {e}")
        return None


def create_great_summary_matrix(great_results: dict | list[dict], n_terms: int = 20) -> pd.DataFrame:
    """Create enrichment summary matrix from GREAT results.
    
    Args:
        great_results: Single result dict or list of dicts for multiple cell types
        n_terms: Number of top terms to keep
    """
    if isinstance(great_results, dict):
        great_results = [great_results]
    
    # Initialize summary lists
    summary_data = []
    
    for result in great_results:
        if result is None or 'binaryHypergeometricGOChartTerms' not in result:
            continue
        
        terms = result.get('binaryHypergeometricGOChartTerms', [])
        for i, term in enumerate(terms[:n_terms]):
            summary_data.append({
                'term_name': term.get('name', ''),
                'term_id': term.get('id', ''),
                'fold_enrichment': term.get('foldEnrichment', 0),
                'adj_pval': term.get('adjPval', 1),
                'neg_log10_padj': -np.log10(max(term.get('adjPval', 1), 1e-300))
            })
    
    return pd.DataFrame(summary_data) if summary_data else pd.DataFrame()


def plot_great_heatmap(enrichment_matrix: pd.DataFrame, cell_types: list[str], 
                       contrasts: list[str], outdir: Path) -> Path:
    """Create heatmap of GREAT enrichment across cell types and contrasts."""
    
    if enrichment_matrix.empty:
        print("⚠ No enrichment data available for heatmap")
        return None
    
    # Pivot for heatmap
    heatmap_data = enrichment_matrix.pivot_table(
        index='term_id',
        columns=['cell_type', 'contrast'],
        values='neg_log10_padj',
        aggfunc='max'
    )
    
    fig, ax = plt.subplots(figsize=(12, max(8, len(heatmap_data) * 0.3)))
    
    # Handle missing data
    heatmap_data = heatmap_data.fillna(0)
    
    sns.heatmap(
        heatmap_data,
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': '-log₁₀(adj. p-value)'},
        linewidths=0.5,
        linecolor='white'
    )
    
    ax.set_title('GREAT enrichment: Human divergent peaks', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Cell type | Contrast', fontsize=11)
    ax.set_ylabel('GREAT term', fontsize=11)
    
    plt.tight_layout()
    
    out_pdf = outdir / 'panel2_great_enrichment_heatmap.pdf'
    out_png = outdir / 'panel2_great_enrichment_heatmap.png'
    
    fig.savefig(out_pdf, bbox_inches='tight', dpi=100)
    fig.savefig(out_png, bbox_inches='tight', dpi=300)
    plt.close(fig)
    
    return out_pdf, out_png

In [24]:
print("Panel 2: Loading significant peaks for GREAT analysis...")
print(f"Focus contrast: {GREAT_FOCUS_CONTRAST}")
print(f"Cell types: {FOREGROUND_CELL_TYPES}")

all_great_results = []

for cell_type in FOREGROUND_CELL_TYPES:
    result_df = load_deseq_result(BRANCH_RESULTS, cell_type, GREAT_FOCUS_CONTRAST)
    
    # Extract significant peaks (positive = Human-high in this contrast)
    sig_peaks = extract_significant_peaks(result_df, annotation_df, direction='positive')
    
    n_peaks = len(sig_peaks)
    print(f"\n{cell_type}: {n_peaks} significant peaks (Human-high)")
    
    if n_peaks == 0:
        continue
    
    # Try to load peak coordinates
    try:
        peak_coords = load_peak_coordinates(sig_peaks['peak_id'].tolist())
        
        if len(peak_coords) > 0:
            print(f"  → Loaded coordinates for {len(peak_coords)} peaks")
            
            # Try GREAT API query (optional, may fail without network)
            request_id = f"{cell_type}_{GREAT_FOCUS_CONTRAST}"
            great_result = query_great_api(peak_coords, request_id=request_id)
            
            if great_result:
                print(f"  ✓ GREAT query successful")
                all_great_results.append({
                    'cell_type': cell_type,
                    'contrast': GREAT_FOCUS_CONTRAST,
                    'n_peaks': n_peaks,
                    'result': great_result
                })
            else:
                print(f"  ⚠ GREAT API unavailable (proceeding with summary)")
        
    except Exception as e:
        print(f"  Error processing peaks: {e}")

print(f"\nCompleted GREAT queries for {len(all_great_results)} cell type(s)")

# Save raw results for reference
great_results_path = PANEL2_OUTDIR / 'great_results.pkl'
import pickle
with open(great_results_path, 'wb') as f:
    pickle.dump(all_great_results, f)
print(f"Saved results to: {great_results_path}")

Panel 2: Loading significant peaks for GREAT analysis...
Focus contrast: Div_Human_vs_Apes
Cell types: ['Macrophages', 'T cells', 'Enterocytes']

Macrophages: 17328 significant peaks (Human-high)

T cells: 23326 significant peaks (Human-high)

Enterocytes: 18276 significant peaks (Human-high)

Completed GREAT queries for 0 cell type(s)
Saved results to: /home/jjanssens/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/21_publication_panels/panel2_great_enrichment/great_results.pkl


In [25]:
print("Panel 2: Creating GREAT enrichment visualization...")

# For now, create a summary table from the results
summary_records = []

for entry in all_great_results:
    cell_type = entry['cell_type']
    contrast = entry['contrast']
    result = entry['result']
    
    # Extract top terms
    summary = create_great_summary_matrix(result, n_terms=15)
    if not summary.empty:
        summary['cell_type'] = cell_type
        summary['contrast'] = contrast
        summary_records.append(summary)

if summary_records:
    enrichment_df = pd.concat(summary_records, ignore_index=True)
    
    # Save enrichment summary
    enrich_path = PANEL2_OUTDIR / 'great_enrichment_summary.tsv'
    enrichment_df.to_csv(enrich_path, sep='\t', index=False)
    print(f"✓ Saved enrichment summary: {enrich_path}")
    print(f"  {len(enrichment_df)} enrichment records across {enrichment_df['cell_type'].nunique()} cell types")
    
    # Create visualization
    # Note: Full heatmap requires pivot, but can also create bar plots for each cell type
    fig, axes = plt.subplots(1, len(FOREGROUND_CELL_TYPES), figsize=(16, 8))
    if len(FOREGROUND_CELL_TYPES) == 1:
        axes = [axes]
    
    for ax, cell_type in zip(axes, FOREGROUND_CELL_TYPES):
        cell_enrich = enrichment_df[enrichment_df['cell_type'] == cell_type].head(12).sort_values('neg_log10_padj')
        
        if not cell_enrich.empty:
            # Shorten term names for readability
            term_labels = [name[:55] + '...' if len(name) > 55 else name 
                          for name in cell_enrich['term_name']]
            
            ax.barh(range(len(cell_enrich)), cell_enrich['neg_log10_padj'], color='#d1495b', alpha=0.8)
            ax.set_yticks(range(len(cell_enrich)))
            ax.set_yticklabels(term_labels, fontsize=9)
            ax.set_xlabel('-log₁₀(adj. p-value)', fontsize=10)
            ax.set_title(f'{cell_type}\n({GREAT_FOCUS_CONTRAST})', fontsize=11, fontweight='bold')
            ax.axvline(-np.log10(0.05), linestyle='--', color='gray', linewidth=1, alpha=0.7)
            ax.grid(axis='x', alpha=0.3)
    
    fig.suptitle('GREAT enrichment: top biological process terms', fontsize=13, fontweight='bold', y=1.00)
    plt.tight_layout()
    
    viz_pdf = PANEL2_OUTDIR / 'panel2_great_enrichment_bars.pdf'
    viz_png = PANEL2_OUTDIR / 'panel2_great_enrichment_bars.png'
    fig.savefig(viz_pdf, bbox_inches='tight', dpi=100)
    fig.savefig(viz_png, bbox_inches='tight', dpi=300)
    plt.close(fig)
    
    print(f"✓ Created enrichment visualization")
    print(f"  PDF: {viz_pdf}")
    print(f"  PNG: {viz_png}")
    
else:
    print("⚠ No GREAT results available for visualization")
    print("  (GREAT API may be unavailable - this is expected without network access)")

Panel 2: Creating GREAT enrichment visualization...
⚠ No GREAT results available for visualization
  (GREAT API may be unavailable - this is expected without network access)


In [29]:
# Alternative: Load pre-computed enrichment results from existing files
print("Loading pre-computed enrichment results...")

ENRICHMENT_DIR = OUTPUT_ROOT / '13_deseq2_R_pseudobulk' / 'differential_results' / 'enrichment'

# Map volcano plot cell type names to enrichment file names
CELL_TYPE_MAPPING = {
    'Macrophages': ['Macrophages', 'Macrophage'],
    'T cells': ['T.cells', 'T_cells', 'T.cell', 'T_cell'],
    'Enterocytes': ['Enterocytes', 'Enterocyte'],
}

enrichment_data = []
loaded_cell_types = []

for cell_type in FOREGROUND_CELL_TYPES:
    # Get possible file name variants for this cell type
    possible_names = CELL_TYPE_MAPPING.get(cell_type, [cell_type])
    
    for file_variant in possible_names:
        fpath = ENRICHMENT_DIR / f'{file_variant}_Human_UP_GO_BP.csv'
        
        if fpath.exists():
            print(f"✓ {cell_type:15} → {fpath.name}")
            
            edf = pd.read_csv(fpath, index_col=0)
            edf = edf.rename(columns={'p.adjust': 'padj'})
            edf['cell_type'] = cell_type  # Use standard name
            edf['neg_log10_padj'] = -np.log10(edf['padj'].clip(lower=1e-300))
            edf = edf.sort_values('padj').head(15)  # Top 15 terms per cell type
            enrichment_data.append(edf)
            loaded_cell_types.append(cell_type)
            break
    else:
        print(f"✗ {cell_type:15} → No pre-computed enrichment (requires external GO tool)")

if enrichment_data:
    all_enrichment = pd.concat(enrichment_data, ignore_index=False)
    
    # Create visualization with subplots for each loaded cell type
    n_cell_types = len(loaded_cell_types)
    fig, axes = plt.subplots(1, n_cell_types, figsize=(5.8 * n_cell_types, 10), sharey=False)
    
    if n_cell_types == 1:
        axes = [axes]
    
    for ax_idx, cell_type in enumerate(loaded_cell_types):
        cell_data = all_enrichment[all_enrichment['cell_type'] == cell_type].sort_values('neg_log10_padj', ascending=True)
        
        if len(cell_data) == 0:
            continue
        
        # Shorten term names for readability
        names = [d[:48] + '...' if len(d) > 48 else d for d in cell_data['Description']]
        
        # Create horizontal bar plot
        axes[ax_idx].barh(range(len(cell_data)), cell_data['neg_log10_padj'], color='#d1495b', alpha=0.8)
        axes[ax_idx].set_yticks(range(len(cell_data)))
        axes[ax_idx].set_yticklabels(names, fontsize=8.5)
        axes[ax_idx].set_xlabel('-log₁₀(adj. p-value)', fontsize=10, fontweight='bold')
        axes[ax_idx].set_title(cell_type, fontsize=11, fontweight='bold', pad=10)
        axes[ax_idx].axvline(-np.log10(0.05), linestyle='--', color='gray', linewidth=0.9, alpha=0.5)
        axes[ax_idx].grid(axis='x', alpha=0.2, linestyle=':')
        
        # Set consistent x-axis limits
        max_val = all_enrichment['neg_log10_padj'].quantile(0.95)
        axes[ax_idx].set_xlim(0, max_val * 1.1)
    
    # Title with data availability note
    title_suffix = f" ({len(loaded_cell_types)}/{len(FOREGROUND_CELL_TYPES)} cell types)" if len(loaded_cell_types) < len(FOREGROUND_CELL_TYPES) else ""
    fig.suptitle(f'GO Biological Process enrichment: Human-divergent peaks{title_suffix}', 
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    # Save outputs
    out_pdf = PANEL2_OUTDIR / 'panel2_enrichment_top_terms.pdf'
    out_png = PANEL2_OUTDIR / 'panel2_enrichment_top_terms.png'
    
    fig.savefig(out_pdf, bbox_inches='tight', dpi=100)
    fig.savefig(out_png, bbox_inches='tight', dpi=300)
    plt.close(fig)
    
    print(f"\n✓ Visualization created:")
    print(f"  PDF: {out_pdf.name}")
    print(f"  PNG: {out_png.name}")
    print(f"  Cell types: {', '.join(loaded_cell_types)}")
    
    # Save summary table
    summary_out = PANEL2_OUTDIR / 'enrichment_summary.tsv'
    summary_cols = ['Description', 'FoldEnrichment', 'padj', 'neg_log10_padj', 'cell_type']
    all_enrichment[summary_cols].to_csv(summary_out, sep='\t')
    print(f"  Summary: {summary_out.name} ({len(all_enrichment)} terms)")
    
else:
    print("⚠ No pre-computed enrichment files found for selected cell types")

Loading pre-computed enrichment results...
✗ Macrophages     → No pre-computed enrichment (requires external GO tool)
✗ T cells         → No pre-computed enrichment (requires external GO tool)
✓ Enterocytes     → Enterocytes_Human_UP_GO_BP.csv

✓ Visualization created:
  PDF: panel2_enrichment_top_terms.pdf
  PNG: panel2_enrichment_top_terms.png
  Cell types: Enterocytes
  Summary: enrichment_summary.tsv (15 terms)


In [28]:
# Fallback: Generate enrichment for missing cell types using gseapy or basic functional annotation
print("\nAttempting to generate enrichment for missing cell types...")

missing_cell_types = [ct for ct in FOREGROUND_CELL_TYPES if ct not in loaded_cell_types]

if missing_cell_types:
    try:
        import gseapy as gp
        from gseapy import Organism
        
        for cell_type in missing_cell_types:
            # Get significant peaks (top genes by log2FC)
            result_df = load_deseq_result(BRANCH_RESULTS, cell_type, GREAT_FOCUS_CONTRAST)
            sig_peaks = extract_significant_peaks(result_df, annotation_df, direction='positive')
            
            # Extract gene names from annotations
            gene_names = sig_peaks['nearest_gene'].dropna().unique()
            gene_names = [g for g in gene_names if isinstance(g, str) and g not in {'<NA>', 'nan', '.'}]
            
            if len(gene_names) > 50:
                print(f"  {cell_type}: {len(gene_names)} genes - running enrichment...")
                
                try:
                    # Run EnrichR enrichment
                    enr = gp.enrichr(
                        gene_list=gene_names[:500],  # Limit to top 500
                        organism='Human',
                        gene_sets=['GO_Biological_Process_2021'],
                        cutoff=0.05
                    )
                    
                    if enr is not None and len(enr.results) > 0:
                        enr_df = enr.results.copy()
                        enr_df['cell_type'] = cell_type
                        enr_df['neg_log10_padj'] = -np.log10(enr_df['Adjusted P-value'].clip(lower=1e-300))
                        enr_df = enr_df.rename(columns={'Term': 'Description'})
                        enr_df = enr_df.head(15)
                        enrichment_data.append(enr_df)
                        loaded_cell_types.append(cell_type)
                        print(f"    ✓ Generated enrichment ({len(enr_df)} terms)")
                except Exception as e:
                    print(f"    ⚠ EnrichR failed: {e}")
            else:
                print(f"  {cell_type}: Only {len(gene_names)} genes (skipping enrichment)")
                
    except ImportError:
        print("  ⚠ gseapy not installed, skipping dynamic enrichment")

print(f"\nTotal cell types with enrichment data: {len(loaded_cell_types)}")


Attempting to generate enrichment for missing cell types...


/local1/scratch/jjans/miniforge3/envs/scenicplus_scenicplus10a2_py311/lib/python3.11/site-packages/bioservices/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


  ⚠ gseapy not installed, skipping dynamic enrichment

Total cell types with enrichment data: 1
